# Thesis supervision letters: CSV and Excel input

1. Put the thesis distribution files in `INPUT_FOLDER` (or list them in `INPUT_FILES`). Works with `.csv`, `.xlsx`, `.xlsm` and old `.xls`, mixed freely. Every sheet of an Excel file is checked; sheets that aren't a thesis roster are skipped.
2. Columns are found by header name: **Group, ID, Name, Batch** plus either **Faculty** or semester columns like **Spring 26 / Summer 26 / Fall 26**. Extra columns are ignored.
3. Semester columns: if there is no Faculty column, the cell holds the faculty for that semester; if there is one, a filled cell means the group is in that semester. Without semester columns, the semester comes from a `Semester` column, then the sheet name, then the file name.
4. Run the cell, pick the semester(s); only the picked semester's column is used. Then enter the date and a reference prefix per semester.
5. Employee IDs: `EMPLOYEE_FILE`, or any sheet with an `Employee ID` column is picked up automatically.
6. Output: `letters/<Semester>/<Faculty>.pdf`.

In [ ]:
import csv
import glob
import os
import re
import subprocess
import sys
from difflib import get_close_matches

for _mod, _pkg in (("reportlab", "reportlab"), ("openpyxl", "openpyxl"), ("xlrd", "xlrd")):
    try:
        __import__(_mod)
    except ImportError:
        print(f"{_pkg} not found -- installing it now (needs internet access)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import openpyxl
import xlrd   # only for old .xls files; .xlsx/.xlsm go through openpyxl
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, KeepTogether
)

# Optional: only mount Drive if running inside Google Colab.
try:
    from google.colab import drive, files  # noqa: F401
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------------------
# 0. Config -- EDIT THESE for your environment
# ---------------------------------------------------------------------------
# Folder holding the thesis distribution files. Every .csv / .xlsx / .xlsm /
# .xls in it is read; in Excel files EVERY sheet is checked, and sheets that
# don't look like a thesis roster (e.g. notes, summaries) are skipped.
# Columns are matched by HEADER NAME, never by position: Group, ID, Name,
# Batch, Faculty, and optionally Semester. Extra columns are ignored.
INPUT_FOLDER = "/content/drive/MyDrive/thesis_data"

# Or list the files explicitly instead (leave empty to use INPUT_FOLDER).
INPUT_FILES = [
    # "/content/drive/MyDrive/thesis_data/thesis_2026.xlsx",
    # "/content/drive/MyDrive/thesis_data/eve_batch.csv",
]

# Faculty name -> Employee ID roster: a .csv, .xlsx or .xls. Optional --
# any sheet inside the input files with an 'Employee ID' column and a
# faculty-name column (and no student columns) is picked up automatically,
# like the 'Employee ID' sheet of the old single workbook.
EMPLOYEE_FILE = "/content/drive/MyDrive/thesis_data/employee_id.csv"
EMPLOYEE_SHEET_NAME = "Employee ID"   # only used when EMPLOYEE_FILE is Excel

OUT_DIR = "/kaggle/working/letters" if os.path.isdir("/kaggle/working") else "letters"
FIXED_CR = 1.0
FIXED_HRS = 1.0

# Accepted header spellings (compared after lower-casing and dropping
# everything that is not a letter or digit, so "Student ID", "student_id"
# and "STUDENT-ID" all become "studentid").
COLUMN_ALIASES = {
    "Group":    ["group", "groupno", "groupnumber", "grp", "grpno"],
    "ID":       ["id", "studentid", "stdid", "sid", "studentidno"],
    "Name":     ["name", "studentname", "nameofstudent", "nameofthestudent"],
    "Batch":    ["batch", "batchno", "batchnumber"],
    "Faculty":  ["faculty", "facultyname", "supervisor", "supervisorname",
                 "facultymember"],
    "Semester": ["semester", "semestername", "session", "term"],
}
REQUIRED_COLUMNS = ["Group", "ID", "Name", "Batch"]   # + Faculty or semester columns
EMP_ALIASES = {
    "EmpID": ["employeeid", "empid", "employeeidno", "id"],
    "FacName": ["facultyname", "name", "faculty", "employeename", "nameoffaculty"],
}
HEADER_SEARCH_ROWS = 15   # header may sit below a few title rows
INPUT_EXTS = (".csv", ".xlsx", ".xlsm", ".xls")

os.makedirs(OUT_DIR, exist_ok=True)


def s(x):
    """Trimmed string version of a cell value, '' for None/blank."""
    if x is None:
        return ""
    return str(x).strip()


def safe_filename(name):
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def clean_emp_id(x):
    """Normalize an ID-like cell. Excel stores 150271 as 150271.0 and some CSV
    exports write '150271.0' too -- strip that '.0'. IDs with leading zeros
    (e.g. '090104') are kept exactly as written."""
    if x is None:
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    x = str(x).strip()
    if re.fullmatch(r"\d+\.0+", x):
        x = x.split(".")[0]
    return x


def normalize_name(n):
    """Case/whitespace/punctuation-insensitive key for name matching."""
    n = n.lower().strip()
    n = n.replace(".", "")
    n = re.sub(r"\s+", " ", n)
    return n


def strip_acronym(n):
    """Drop a trailing '(XYZ)' style acronym, e.g. 'Md. Rezaul Karim (RK)'
    -> 'Md. Rezaul Karim'."""
    return re.sub(r"\s*\([^)]*\)\s*$", "", n).strip()


def norm_header(h):
    return re.sub(r"[^a-z0-9]", "", s(h).lower())


SEASONS = {"spring": "Spring", "sp": "Spring", "summer": "Summer", "su": "Summer",
           "fall": "Fall", "fa": "Fall", "fl": "Fall", "autumn": "Autumn",
           "winter": "Winter"}
SEMESTER_RE = re.compile(
    r"(?<![a-z])(spring|summer|fall|autumn|winter|sp|su|fa|fl)[\s_\-]*'?(\d{4}|\d{2})(?!\d)",
    re.I)


def normalize_semester(text):
    """'summer-2026', 'SUMMER 26', 'Summer2026', 'SU26' -> 'Summer 2026'.
    Anything that doesn't look like <season> <year> is kept as written."""
    text = s(text)
    m = SEMESTER_RE.search(text)
    if not m:
        return re.sub(r"\s+", " ", text)
    year = m.group(2)
    if len(year) == 2:
        year = "20" + year
    return f"{SEASONS[m.group(1).lower()]} {year}"


def find_semester(text):
    m = SEMESTER_RE.search(s(text))
    return normalize_semester(m.group(0)) if m else ""


def is_semester_header(h):
    """True for a column header that is JUST a semester, e.g. 'Spring 26',
    'Summer-2026', 'Fall26', 'SU26' -- not 'Remarks Spring 26'."""
    h = s(h)
    m = SEMESTER_RE.fullmatch(h.strip()) if h else None
    return bool(m)


# ---------------------------------------------------------------------------
# 1. File readers -- every input becomes a list of "tables" (one per CSV,
#    one per Excel sheet), each a plain list of rows.
# ---------------------------------------------------------------------------
def read_csv_rows(path):
    """All rows of a CSV as lists of strings. Excel on Windows saves CSVs as
    either UTF-8 with BOM or cp1252, so try both."""
    for enc in ("utf-8-sig", "cp1252"):
        try:
            with open(path, newline="", encoding=enc) as fh:
                return [row for row in csv.reader(fh)]
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode '{path}' as UTF-8 or cp1252.")


def read_xlsx_sheets(path):
    """{sheet_name: rows} for .xlsx/.xlsm. Merged cells are un-merged: every
    cell of a merged block gets the block's value, so a Faculty or Group
    merged over 4 student rows shows up on all 4."""
    wb = openpyxl.load_workbook(path, data_only=True)
    out = {}
    for ws in wb.worksheets:
        rows = [list(r) for r in ws.iter_rows(values_only=True)]
        for rng in ws.merged_cells.ranges:
            val = ws.cell(rng.min_row, rng.min_col).value
            for r in range(rng.min_row, rng.max_row + 1):
                for c in range(rng.min_col, rng.max_col + 1):
                    if r - 1 < len(rows) and c - 1 < len(rows[r - 1]):
                        rows[r - 1][c - 1] = val
        out[ws.title] = rows
    return out


def read_xls_sheets(path):
    """{sheet_name: rows} for old .xls files, merged cells un-merged too."""
    try:
        book = xlrd.open_workbook(path, formatting_info=True)
    except NotImplementedError:          # some .xls variants lack formatting info
        book = xlrd.open_workbook(path)
    out = {}
    for sh in book.sheets():
        rows = [[sh.cell_value(r, c) for c in range(sh.ncols)] for r in range(sh.nrows)]
        for rlo, rhi, clo, chi in getattr(sh, "merged_cells", []):
            val = rows[rlo][clo]
            for r in range(rlo, rhi):
                for c in range(clo, chi):
                    rows[r][c] = val
        out[sh.name] = rows
    return out


def read_tables(path):
    """[(label, sheet_name, rows)] for any supported file."""
    ext = os.path.splitext(path)[1].lower()
    base = os.path.basename(path)
    if ext == ".csv":
        return [(base, "", read_csv_rows(path))]
    if ext in (".xlsx", ".xlsm"):
        sheets = read_xlsx_sheets(path)
    elif ext == ".xls":
        sheets = read_xls_sheets(path)
    else:
        raise ValueError(f"Unsupported file type: {base}")
    return [(f"{base} [{name}]", name, rows) for name, rows in sheets.items()]


def find_header(rows, aliases, required):
    """Return (header_row_index, {logical_column: col_index}) for the first
    row in the top HEADER_SEARCH_ROWS that contains every required column."""
    for i, row in enumerate(rows[:HEADER_SEARCH_ROWS]):
        normed = [norm_header(c) for c in row]
        col_idx = {}
        for logical, names in aliases.items():
            for j, h in enumerate(normed):
                if h in names:
                    col_idx[logical] = j
                    break
        if all(c in col_idx for c in required):
            return i, col_idx
    return None, None


def find_thesis_header(rows):
    """Header row of a thesis roster: Group/ID/Name/Batch plus either a
    Faculty column or one-or-more semester columns ('Spring 26', ...).
    Returns (row_index, col_idx, {semester: col_index}) or (None, None, None)."""
    for i, row in enumerate(rows[:HEADER_SEARCH_ROWS]):
        hdr, col = find_header([row], COLUMN_ALIASES, REQUIRED_COLUMNS)
        if hdr is None:
            continue
        sem_cols = {}
        for j, h in enumerate(row):
            if is_semester_header(h):
                sem_cols.setdefault(normalize_semester(h), j)
        if "Faculty" in col or sem_cols:
            return i, col, sem_cols
    return None, None, None


# ---------------------------------------------------------------------------
# 2. Employee ID roster
# ---------------------------------------------------------------------------
emp_map = {}           # normalized name (full, incl. acronym) -> employee id
emp_map_stripped = {}  # normalized name with acronym stripped -> employee id


def add_roster_rows(rows, hdr, col):
    n = 0
    for row in rows[hdr + 1:]:
        row = list(row) + [None] * (max(col.values()) + 1)
        emp_id = clean_emp_id(row[col["EmpID"]])
        name = s(row[col["FacName"]])
        if not emp_id or not name:
            continue
        emp_map.setdefault(normalize_name(name), emp_id)
        emp_map_stripped.setdefault(normalize_name(strip_acronym(name)), emp_id)
        n += 1
    return n


def load_employee_file(path, sheet_name):
    if not os.path.isfile(path):
        return
    if path.lower().endswith(".csv"):
        rows = read_csv_rows(path)
    else:
        sheets = (read_xls_sheets(path) if path.lower().endswith(".xls")
                  else read_xlsx_sheets(path))
        if sheet_name not in sheets:
            raise ValueError(f"Sheet '{sheet_name}' not found in '{path}'. "
                             f"Available sheets: {list(sheets)}")
        rows = sheets[sheet_name]
    hdr, col = find_header(rows, EMP_ALIASES, ["EmpID", "FacName"])
    if hdr is None:
        hdr, col = 0, {"EmpID": 0, "FacName": 1}   # old layout: A = ID, B = name
    n = add_roster_rows(rows, hdr, col)
    print(f"Loaded {n} employee ID entries from '{os.path.basename(path)}'.")


def try_load_roster_table(label, rows):
    """Use a sheet/CSV as the employee roster if it has an explicit Employee
    ID header (not just 'ID', which every student sheet has too)."""
    strict = {"EmpID": ["employeeid", "empid", "employeeidno"],
              "FacName": EMP_ALIASES["FacName"]}
    hdr, col = find_header(rows, strict, ["EmpID", "FacName"])
    if hdr is None:
        return False
    n = add_roster_rows(rows, hdr, col)
    print(f"  {label}: employee roster, {n} entries")
    return True


def lookup_employee_id(faculty_name):
    key_full = normalize_name(faculty_name)
    if key_full in emp_map:
        return emp_map[key_full]
    key_stripped = normalize_name(strip_acronym(faculty_name))
    if key_stripped in emp_map_stripped:
        return emp_map_stripped[key_stripped]
    if key_stripped in emp_map:
        return emp_map[key_stripped]
    return ""


# ---------------------------------------------------------------------------
# 3. Turn one table into flat student records
# ---------------------------------------------------------------------------
def parse_thesis_table(label, rows, fallback_semester, from_csv):
    """Return (records, mode_description) or ([], None) if this table is not
    a thesis roster.

    Semester is decided in this order:
      a) semester-named columns ('Spring 26', 'Summer 26', 'Fall 26'): one
         record per student per filled-in semester column. If the table has
         no Faculty column, the cell value IS the faculty for that semester;
         if it has one, a non-blank cell just means "in this semester".
         Only the column of the semester the user picks is ever used.
      b) a 'Semester' column,
      c) the sheet name, then the file name (e.g. 'summer_2026.csv').
    Group / Faculty / Semester are forward-filled down blank cells (what
    merged cells become in a CSV); forward-fill restarts in every table.
    Excel merged cells are already un-merged by the readers, so in Excel a
    blank semester-column cell is a real blank and is NOT filled."""
    hdr, col, sem_cols = find_thesis_header(rows)
    if hdr is None:
        return [], None
    if not sem_cols and "Semester" not in col and not fallback_semester:
        raise ValueError(
            f"{label}: no semester columns, no 'Semester' column and no "
            f"semester in the sheet/file name -- can't tell which semester "
            f"these students belong to.")

    has_faculty = "Faculty" in col
    if sem_cols:
        mode = (f"semester columns {list(sem_cols)} "
                + ("(filter; faculty from 'Faculty' column)" if has_faculty
                   else "(cell value = faculty)"))
    elif "Semester" in col:
        mode = "'Semester' column"
    else:
        mode = f"semester from name: {fallback_semester}"

    width = max(list(col.values()) + list(sem_cols.values())) + 1
    records = []
    cur_group, cur_faculty, cur_sem = "", "", fallback_semester
    cur_semval = {sem: "" for sem in sem_cols}

    for line_no, r in enumerate(rows[hdr + 1:], start=hdr + 2):
        r = list(r) + [None] * width
        grp_str = clean_emp_id(r[col["Group"]])
        id_str = clean_emp_id(r[col["ID"]])
        fac_str = s(r[col["Faculty"]]) if has_faculty else ""
        new_group = ((bool(grp_str) and grp_str != cur_group)
                     or (bool(fac_str) and fac_str != cur_faculty))

        if "Semester" in col and not sem_cols:
            sem_str = normalize_semester(r[col["Semester"]])
            if sem_str:
                cur_sem = sem_str
        if fac_str and fac_str != cur_faculty:
            cur_faculty = fac_str
            if not grp_str and id_str:
                print(f"  WARNING {label} row {line_no}: faculty changes to "
                      f"'{fac_str}' but Group is blank -- student put in the "
                      f"previous group '{cur_group}'.")
        if grp_str:
            cur_group = grp_str
        # Blank semester-column cells:
        #  - inside the same group: always inherit (a group can't be half in
        #    a semester);
        #  - at a new group: blank means "not in this semester" -- except in
        #    a CSV where the cell value is the faculty, because there a name
        #    merged over all of a faculty's groups (like the old Faculty
        #    column) leaves blanks after export. Excel merges are already
        #    un-merged by the readers, so Excel never needs that.
        for sem, j in sem_cols.items():
            v = s(r[j])
            if v:
                cur_semval[sem] = v
            elif new_group and (has_faculty or not from_csv):
                cur_semval[sem] = ""

        # Rows with no Student ID are blank separator rows or section titles
        # (e.g. "Eve Batch 44 (...)") -- skip both.
        if not id_str:
            continue
        if re.fullmatch(r"\d+(\.\d+)?[eE]\+\d+", id_str):
            print(f"  WARNING {label} row {line_no}: Student ID '{id_str}' is in "
                  f"scientific notation -- Excel mangled it. Format the ID "
                  f"column as Text before saving the CSV.")

        base = {
            "Group": cur_group,
            "ID": id_str,
            "Name": s(r[col["Name"]]),
            "Batch": clean_emp_id(r[col["Batch"]]),
            "Source": label,
            "Line": line_no,
        }
        if sem_cols:
            for sem in sem_cols:
                v = cur_semval[sem]
                if not v:
                    continue
                faculty = cur_faculty if has_faculty else v
                if faculty:
                    records.append(dict(base, Semester=sem, Faculty=faculty))
        elif cur_faculty:
            records.append(dict(base, Semester=cur_sem or "(no semester)",
                                Faculty=cur_faculty))
    return records, mode


# ---------------------------------------------------------------------------
# 4. Collect the input files and read them all
# ---------------------------------------------------------------------------
def is_input_file(p):
    b = os.path.basename(p)
    return b.lower().endswith(INPUT_EXTS) and not b.startswith("~$")


if INPUT_FILES:
    input_paths = list(INPUT_FILES)
else:
    input_paths = sorted((p for p in glob.glob(os.path.join(INPUT_FOLDER, "*"))
                          if is_input_file(p)), key=str.lower)

if not input_paths and IN_COLAB:
    print(f"No input files found in '{INPUT_FOLDER}'. Upload them now "
          f"(CSV or Excel, you can select several at once):")
    uploaded = files.upload()
    input_paths = sorted(p for p in uploaded if is_input_file(p))

if not input_paths:
    raise FileNotFoundError(
        f"No CSV/Excel files found. Put them in INPUT_FOLDER ('{INPUT_FOLDER}') "
        f"or list them in INPUT_FILES.")

load_employee_file(EMPLOYEE_FILE, EMPLOYEE_SHEET_NAME)
emp_abs = os.path.abspath(EMPLOYEE_FILE)

all_records = []
print(f"\nReading {len(input_paths)} file(s):")
for p in input_paths:
    if os.path.abspath(p) == emp_abs:
        continue
    for label, sheet_name, rows in read_tables(p):
        fallback = find_semester(sheet_name) or find_semester(os.path.basename(p))
        recs, mode = parse_thesis_table(label, rows, fallback,
                                        p.lower().endswith(".csv"))
        if mode is None:
            if not try_load_roster_table(label, rows):
                print(f"  SKIPPED {label}: no Group/ID/Name/Batch + Faculty "
                      f"header found")
            continue
        sems = sorted({r["Semester"] for r in recs})
        print(f"  {label:45s} {len(recs):4d} row(s)  [{mode}]  "
              f"semester(s): {', '.join(sems) or '-'}")
        all_records.extend(recs)

if not all_records:
    raise ValueError("No student rows found in any file -- see the SKIPPED "
                     "reasons above.")
if not emp_map:
    print("WARNING: no employee roster found -- every letter will get the "
          "'____' Employee ID placeholder.")

# ---------------------------------------------------------------------------
# 5. Let the user pick the semester(s)
# ---------------------------------------------------------------------------
sem_counts = {}
for r in all_records:
    sem_counts[r["Semester"]] = sem_counts.get(r["Semester"], 0) + 1
semesters = list(sem_counts)

print("\nSemesters found:")
for i, sem in enumerate(semesters, start=1):
    print(f"  {i}. {sem:25s} ({sem_counts[sem]} students)")

while True:
    choice = input("Select semester(s) -- number(s) like '1' or '1,3', or 'all': ").strip()
    if choice.lower() == "all":
        selected = semesters
        break
    try:
        picks = [int(c) for c in re.split(r"[,\s]+", choice) if c]
        if picks and all(1 <= n <= len(semesters) for n in picks):
            selected = [semesters[n - 1] for n in dict.fromkeys(picks)]
            break
    except ValueError:
        pass
    print(f"  Invalid choice '{choice}'. Enter numbers between 1 and {len(semesters)}.")

LETTER_DATE = input("Enter Letter Date (e.g., 10/05/2026): ")
REF_PREFIXES = {}
for sem in selected:
    REF_PREFIXES[sem] = input(
        f"Enter Reference Prefix for {sem} (e.g., UU/EEE/Thesis/OL/SU26/): ")


# ---------------------------------------------------------------------------
# 6. Collapse rows into groups, per faculty, for one semester
# ---------------------------------------------------------------------------
def build_faculty_groups(records):
    """Return (faculty_order, faculty_map). Groups are keyed by (source,
    group no) so 'Group 1' in two different files/sheets never gets merged
    into one group by accident. A student ID that appears twice in the same
    semester is kept once (first occurrence) and reported."""
    faculty_order = []
    faculty_map = {}
    seen_ids = {}

    for rec in records:
        if rec["ID"] in seen_ids:
            first = seen_ids[rec["ID"]]
            print(f"  WARNING duplicate student {rec['ID']} ({rec['Name']}): "
                  f"{rec['Source']} row {rec['Line']} ignored, already in "
                  f"{first['Source']} row {first['Line']} under {first['Faculty']}.")
            continue
        seen_ids[rec["ID"]] = rec

        f = rec["Faculty"]
        if f not in faculty_map:
            faculty_map[f] = []
            faculty_order.append(f)

        groups = faculty_map[f]
        key = (rec["Source"], rec["Group"])
        match = next((g for g in groups if g["Key"] == key), None)
        if match is None:
            groups.append({
                "Key": key,
                "GroupNo": rec["Group"],
                "IDs": [rec["ID"]],
                "Names": [rec["Name"]],
                "Batch": [rec["Batch"]],
            })
        else:
            match["IDs"].append(rec["ID"])
            match["Names"].append(rec["Name"])
            match["Batch"].append(rec["Batch"])

    return faculty_order, faculty_map


# ---------------------------------------------------------------------------
# 7. PDF generation (one file per faculty) -- layout unchanged
# ---------------------------------------------------------------------------
def build_styles(table_font_size, body_font_size):
    return {
        "title": ParagraphStyle("title", fontName="Times-Bold", fontSize=12,
                                 alignment=TA_CENTER, leading=14),
        "title_u": ParagraphStyle("title_u", fontName="Times-Bold", fontSize=12,
                                   alignment=TA_CENTER, leading=14),
        "body": ParagraphStyle("body", fontName="Times-Roman", fontSize=body_font_size,
                                leading=body_font_size + 3),
        "cell": ParagraphStyle("cell", fontName="Times-Roman", fontSize=table_font_size,
                            alignment=TA_CENTER, leading=table_font_size),
        "cell_left": ParagraphStyle("cell_left", fontName="Times-Roman", fontSize=table_font_size,
                                alignment=TA_LEFT, leading=table_font_size),
        "cell_header": ParagraphStyle("cell_header", fontName="Times-Bold", fontSize=table_font_size,
                                  alignment=TA_CENTER, leading=table_font_size),
        "sig": ParagraphStyle("sig", fontName="Times-Bold", fontSize=body_font_size,
                               alignment=TA_CENTER, leading=body_font_size + 3),
    }


def generate_faculty_pdf(faculty_name, groups, emp_id, out_path,
                         semester_text, ref_prefix, letter_date):
    n_students = sum(len(g["IDs"]) for g in groups)

    if n_students <= 20:
        table_font_size, body_font_size = 9, 10
    elif n_students <= 30:
        table_font_size, body_font_size = 8, 9
    else:
        table_font_size, body_font_size = 7, 9

    styles = build_styles(table_font_size, body_font_size)

    doc = SimpleDocTemplate(
        out_path, pagesize=A4,
        leftMargin=0.6 * 72, rightMargin=0.6 * 72,
        topMargin=0.5 * 72, bottomMargin=0.5 * 72,
    )
    usable_width = doc.width

    story = []
    story.append(Paragraph("Department of Electrical and Electronic Engineering", styles["title"]))
    story.append(Paragraph("Uttara University", styles["title"]))
    story.append(Paragraph("<u>Claim of Thesis/Project Supervision</u>", styles["title_u"]))
    story.append(Spacer(1, 8))

    ref_table = Table(
        [[Paragraph(f"Ref: {ref_prefix}", styles["body"]),
          Paragraph(f"Date: {letter_date}", styles["body"])]],
        colWidths=[usable_width * 0.65, usable_width * 0.3],
    )
    ref_table.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ]))
    story.append(ref_table)
    story.append(Spacer(1, 8))

    story.append(Paragraph("To,", styles["body"]))
    emp_id_text = emp_id if emp_id else "____"
    story.append(Paragraph(
        f"{faculty_name}" + "&nbsp;" * 12 + f"Employee ID: {emp_id_text}", styles["body"]))
    story.append(Paragraph(
        f"Sub: Invitation letter to conduct Thesis/Project (EEE 4000) for {semester_text}",
        styles["body"]))
    story.append(Spacer(1, 6))
    story.append(Paragraph("Dear Sir/Madam,", styles["body"]))
    story.append(Spacer(1, 6))

    intro = (
        "We are pleased to inform you that, you are nominated to conduct the course of "
        "&quot;Thesis/Project (EEE 4000)&quot; for " + semester_text + ". The following "
        "students will be guided under your supervision in the concerned department as "
        "per above schedule."
    )
    story.append(Paragraph(intro, styles["body"]))
    story.append(Spacer(1, 8))

    # ---- Table: one physical row per STUDENT, with Sl.No/Group No/Cr./Hrs.
    #      genuinely merged (SPAN) across each group's row range.
    headers = ["Sl.No", "Group No", "Cr.", "Hrs.", "Student ID", "Student Name", "Batch"]
    table_data = [[Paragraph(h, styles["cell_header"]) for h in headers]]

    span_cmds = []
    total_cr = 0.0
    total_hrs = 0.0
    row_idx = 1  # row 0 is the header

    for g_idx, g in enumerate(groups, start=1):
        n_rows = len(g["IDs"])
        total_cr += FIXED_CR
        total_hrs += FIXED_HRS
        start_row = row_idx

        for r in range(n_rows):
            sl_txt = str(g_idx) if r == 0 else ""
            grp_txt = g["GroupNo"] if r == 0 else ""
            cr_txt = f"{FIXED_CR:.1f}" if r == 0 else ""
            hrs_txt = f"{FIXED_HRS:.1f}" if r == 0 else ""
            table_data.append([
                Paragraph(sl_txt, styles["cell"]),
                Paragraph(grp_txt, styles["cell"]),
                Paragraph(cr_txt, styles["cell"]),
                Paragraph(hrs_txt, styles["cell"]),
                Paragraph(g["IDs"][r], styles["cell"]),
                Paragraph(g["Names"][r].upper(), styles["cell_left"]),
                Paragraph(g["Batch"][r], styles["cell"]),
            ])
            row_idx += 1

        end_row = row_idx - 1
        if n_rows > 1:
            for col in range(4):  # Sl.No, Group No, Cr., Hrs.
                span_cmds.append(("SPAN", (col, start_row), (col, end_row)))
            span_cmds.append(("VALIGN", (0, start_row), (3, end_row), "MIDDLE"))

    # Total row
    table_data.append([
        Paragraph("<b>Total=</b>", styles["cell"]),
        Paragraph("", styles["cell_header"]),
        Paragraph(f"{total_cr:g}", styles["cell_header"]),
        Paragraph(f"{total_hrs:g}", styles["cell_header"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
    ])
    total_row_idx = row_idx
    span_cmds.append(("SPAN", (0, total_row_idx), (1, total_row_idx)))
    span_cmds.append(("ALIGN", (1, total_row_idx), (1, total_row_idx), "RIGHT"))
    span_cmds.append(("SPAN", (4, total_row_idx), (6, total_row_idx)))

    span_cmds.append(("LINEBELOW", (4, total_row_idx), (6, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBELOW", (0, total_row_idx), (1, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBEFORE", (0, total_row_idx), (1, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEAFTER", (6, total_row_idx), (6, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEBELOW", (2, total_row_idx), (3, total_row_idx), 1, colors.black))

    col_fractions = (0.08, 0.12, 0.08, 0.08, 0.14, 0.40, 0.10)
    col_widths = [usable_width*.90 * f for f in col_fractions]
    main_table = Table(table_data, colWidths=col_widths, repeatRows=1)
    style_cmds = [
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),
        ("RIGHTPADDING", (0, 0), (-1, -1), 2),
        ("TOPPADDING", (0, 0), (-1, -1), 1),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 1),
    ] + span_cmds
    main_table.setStyle(TableStyle(style_cmds))
    story.append(main_table)
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Total Amount of Claim:&nbsp;&nbsp;Tk:</b>", styles["body"]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("Thanking you very much for your cooperation.", styles["body"]))
    story.append(Paragraph("Yours Sincerely", styles["body"]))
    story.append(Spacer(1, 28))
    story.append(Paragraph("Thesis/Project Convener, Dept. of EEE, Uttara University", styles["body"]))
    story.append(Spacer(1, 28))

    sig_table = Table([[
        Paragraph("<br/>Signature of the Faculty Member<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Coordinator<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Chairman<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 3] * 3)
    sig_table.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP")]))
    story.append(sig_table)
    story.append(Spacer(1, 28))

    sig_table2 = Table([[
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 2] * 2)

    checked_by_section = [
        Paragraph("<b>Checked By:</b>", styles["body"]),
        Spacer(1, 28),
        sig_table2,
        Spacer(1, 28)
    ]

    story.append(KeepTogether(checked_by_section))

    doc.build(story)


# ---------------------------------------------------------------------------
# 8. Generate the letters: one folder per selected semester, one PDF per
#    faculty inside it. Only the selected semester's records are used.
# ---------------------------------------------------------------------------
all_unmatched = {}
for sem in selected:
    sem_records = [r for r in all_records if r["Semester"] == sem]
    print(f"\n=== {sem}: {len(sem_records)} student rows ===")
    faculty_order, faculty_map = build_faculty_groups(sem_records)
    print(f"Found {len(faculty_order)} faculty members.")

    sem_dir = os.path.join(OUT_DIR, safe_filename(sem))
    os.makedirs(sem_dir, exist_ok=True)

    for f in faculty_order:
        groups = faculty_map[f]
        n_students = sum(len(g["IDs"]) for g in groups)
        emp_id = lookup_employee_id(f)
        if not emp_id:
            all_unmatched.setdefault(f, []).append(sem)

        # Sanity check: every group having exactly one student usually means
        # the Group column is wrong / not forward-filled.
        if n_students > 1 and len(groups) == n_students:
            print(f"  WARNING {f}: {len(groups)} groups for {n_students} students "
                  f"-- check the Group column for this faculty.")

        out_name = safe_filename(f) + ".pdf"
        generate_faculty_pdf(f, groups, emp_id, os.path.join(sem_dir, out_name),
                             sem, REF_PREFIXES[sem], LETTER_DATE)
        print(f"  {f:45s} {len(groups):2d} group(s), {n_students:3d} student(s)  "
              f"Emp ID: {emp_id or 'NOT FOUND'}  -> {out_name}")

if all_unmatched:
    all_names = list(emp_map)
    print(f"\nWARNING: no Employee ID for {len(all_unmatched)} faculty member(s) -- "
          f"their letters keep the '____' placeholder:")
    for f, sems in all_unmatched.items():
        guess = get_close_matches(normalize_name(f), all_names, n=1, cutoff=0.6)
        hint = f"  (closest roster name: '{guess[0]}')" if guess else ""
        print(f"    - {f}  [{', '.join(sems)}]{hint}")

print(f"\nAll letters written to '{OUT_DIR}' (one sub-folder per semester).")

if IN_COLAB:
    subprocess.check_call(["zip", "-r", "letters.zip", OUT_DIR])
    files.download('letters.zip')
